In [ ]:
"""
Figure: Methods Pipeline (swim-lane diagram with explicit execution sequence)
============================================================================

Renders the 21-phase ComputationalReview pipeline as a swim-lane diagram,
faithfully reproducing the actor/critic/validator separation defined in
skills/comprev-orchestrator-v29.md (Phase Index).  An explicit numbered
arrow snake traces the actual execution order (34 steps), and a step-number
badge in each box's corner identifies its position in that sequence.

Lanes (top to bottom):
  - Coordinator   -- routing across phases; gates between phases (dotted rail)
  - DATAML agent  -- mechanical actors (upper sub-row) + validators (lower sub-row)
  - LITREVIEW agent -- scientific actors and blinded critics

Element types (per Phase Index):
  - ACTOR boxes    -- solid coloured by lane (DATAML blue / LITREVIEW orange)
  - CRITIC boxes   -- LITREVIEW phases with an information barrier from the actor;
                      dashed border + diagonal hatch.  Phases 6, 8, 12, 16.
  - VALIDATOR boxes -- DATAML phases that re-check an actor's output, with the same
                       information barrier from the actor; dashed border + diagonal
                       hatch.  Phases 1V, 2V, 3V, 5V, 7V, 9V, 14V, 15V, 17V, 19V, 20V,
                       and Phase 21 (deploy-polish validator, no separate actor).

Critics and validators share the same visual convention -- they are the two
flavours of info-barriered reviewer (scientific vs mechanical); lane colour
distinguishes which.

Special cases:
  - Phase 1 has TWO actors (LITREVIEW scoping + DATAML materialise) at the
    same x, stacked across lanes.  They run in parallel; the snake visits
    LITREVIEW first then DATAML for diagram clarity.
  - Phase 20a (Methods Ledger Refresh) runs between Phases 19V and 20.
  - Phase 21 (Deploy Polish) has no separate actor; its frame is both actor
    and validator, reading the deployed Pages artifact.
"""


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Circle

# ---------------------------------------------------------------------------
# Authoritative phase table -- (x_slot, label, role, kind, displayed_phase_id)
# ---------------------------------------------------------------------------
ACTORS = [
    ( 1, "Scope",                  "expert", "actor",  "1"),
    ( 1, "Materialise",            "dataml", "actor",  "1"),
    ( 2, "Evidence\nGathering",    "expert", "actor",  "2"),
    ( 3, "Citation\nInfra",        "dataml", "actor",  "3"),
    ( 4, "Scaffold",               "expert", "actor",  "4"),
    ( 5, "Evidence\nCuration",     "dataml", "actor",  "5"),
    ( 6, "Figure\nAudit",          "expert", "critic", "6"),
    ( 7, "Section\nDrafting",      "expert", "actor",  "7"),
    ( 8, "Section\nCritics",       "expert", "critic", "8"),
    ( 9, "Bibliography",           "dataml", "actor",  "9"),
    (10, "Integration",            "expert", "actor",  "10"),
    (11, "Intro /\nConclusion",    "expert", "actor",  "11"),
    (12, "Bookend\nCritic",        "expert", "critic", "12"),
    (13, "Methods",                "dataml", "actor",  "13"),
    (14, "Document\nAssembly",     "dataml", "actor",  "14"),
    (15, "Citation\nTriples",      "dataml", "actor",  "15"),
    (16, "Citation\nVerification", "expert", "critic", "16"),
    (17, "Fix\nPreparation",       "dataml", "actor",  "17"),
    (18, "Fix\nExecution",         "expert", "actor",  "18"),
    (19, "Fix\nApplication",       "dataml", "actor",  "19"),
    (20, "Methods\nLedger\nRefresh","dataml","actor",  "20a"),
    (21, "Repository\nPush",       "dataml", "actor",  "20"),
]

VALIDATORS = [
    ( 1, "1V" ),
    ( 2, "2V" ),
    ( 3, "3V" ),
    ( 5, "5V" ),
    ( 7, "7V" ),
    ( 9, "9V" ),
    (14, "14V"),
    (15, "15V"),
    (17, "17V"),
    (19, "19V"),
    (21, "20V"),
    (22.4, "Deploy\nPolish (21)"),
]

# Execution order: (x_slot, lane_key) -> step number (1-indexed).  This drives
# both the arrow path and the corner badges.
EXEC_SEQUENCE = [
    ( 1, "expert"),       ( 1, "dataml_actor"), ( 1, "dataml_val"),
    ( 2, "expert"),       ( 2, "dataml_val"),
    ( 3, "dataml_actor"), ( 3, "dataml_val"),
    ( 4, "expert"),
    ( 5, "dataml_actor"), ( 5, "dataml_val"),
    ( 6, "expert"),
    ( 7, "expert"),       ( 7, "dataml_val"),
    ( 8, "expert"),
    ( 9, "dataml_actor"), ( 9, "dataml_val"),
    (10, "expert"),
    (11, "expert"),
    (12, "expert"),
    (13, "dataml_actor"),
    (14, "dataml_actor"),(14, "dataml_val"),
    (15, "dataml_actor"),(15, "dataml_val"),
    (16, "expert"),
    (17, "dataml_actor"),(17, "dataml_val"),
    (18, "expert"),
    (19, "dataml_actor"),(19, "dataml_val"),
    (20, "dataml_actor"),
    (21, "dataml_actor"),(21, "dataml_val"),
    (22.4, "dataml_val"),
]
STEP_OF = {pos: i + 1 for i, pos in enumerate(EXEC_SEQUENCE)}

# ---------------------------------------------------------------------------
# Style
# ---------------------------------------------------------------------------
EDGE = {"coord": "#2e7d32", "dataml": "#1565c0", "expert": "#e65100"}
FILL = {"coord": "#c8e6c9", "dataml": "#bbdefb", "expert": "#ffe0b2"}
VALIDATOR_FILL = "#e3f2fd"
CRITIC_EDGE = "#6a1b9a"
DASH = (0, (4, 2))

# Reviewer "stamp" --- a solid color bar across the top of critic/validator
# boxes, replacing the harder-to-read diagonal hatch.  Each colour matches
# its reviewer family.
STAMP_HEIGHT_FRAC = 0.22    # fraction of box height occupied by the top bar
ARROW_COLOR = "#37474f"   # charcoal -- darker than before for visibility
BADGE_FILL = "#263238"
BADGE_TEXT = "white"

LANE_Y = {"coord": 4.6, "dataml_actor": 2.85, "dataml_val": 1.65, "expert": 0.2}
LANE_LABEL = {
    "coord":  "COORDINATOR\n(routing + gates)",
    "dataml": "DATAML AGENT\n(actors  /  validators)",
    "expert": "LITREVIEW AGENT\n(actors  /  critics)",
}
DATAML_LANE_TOP = LANE_Y["dataml_actor"] + 0.75
DATAML_LANE_BOTTOM = LANE_Y["dataml_val"] - 0.75
DATAML_LANE_CENTER = (DATAML_LANE_TOP + DATAML_LANE_BOTTOM) / 2

# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(22, 8.4))
ax.set_xlim(-2.6, 24.0)
ax.set_ylim(-1.4, 6.0)
ax.axis("off")

box_w, box_h = 0.93, 0.92

# Lane backgrounds
ax.add_patch(FancyBboxPatch(
    (-1.9, LANE_Y["coord"] - 0.65), 25.6, 1.30,
    boxstyle="round,pad=0.02",
    facecolor=FILL["coord"], edgecolor="none", alpha=0.20, zorder=0,
))
ax.text(-1.8, LANE_Y["coord"], LANE_LABEL["coord"],
        ha="left", va="center", fontsize=8.6, weight="bold",
        color=EDGE["coord"], zorder=1)

ax.add_patch(FancyBboxPatch(
    (-1.9, DATAML_LANE_BOTTOM), 25.6, DATAML_LANE_TOP - DATAML_LANE_BOTTOM,
    boxstyle="round,pad=0.02",
    facecolor=FILL["dataml"], edgecolor="none", alpha=0.20, zorder=0,
))
ax.text(-1.8, DATAML_LANE_CENTER, LANE_LABEL["dataml"],
        ha="left", va="center", fontsize=8.6, weight="bold",
        color=EDGE["dataml"], zorder=1)

ax.add_patch(FancyBboxPatch(
    (-1.9, LANE_Y["expert"] - 0.75), 25.6, 1.50,
    boxstyle="round,pad=0.02",
    facecolor=FILL["expert"], edgecolor="none", alpha=0.20, zorder=0,
))
ax.text(-1.8, LANE_Y["expert"], LANE_LABEL["expert"],
        ha="left", va="center", fontsize=8.6, weight="bold",
        color=EDGE["expert"], zorder=1)

# Coordinator gate rail (dotted)
GATE_Y = LANE_Y["coord"] - 0.75
ax.plot([0.5, 22.5], [GATE_Y, GATE_Y], color="#8a8a8a",
        linewidth=0.7, linestyle=(0, (1, 2)), zorder=1)
ax.text(22.7, GATE_Y, "  gate", ha="left", va="center", fontsize=7,
        color="#666", style="italic")

# ---------------------------------------------------------------------------
# Snake the execution-order arrows.  Use FancyArrowPatch with explicit
# data-coordinate endpoints offset by box-half-size (works regardless of dpi).
# Drawn first so boxes overlay arrowheads at the box edges.
# ---------------------------------------------------------------------------
from matplotlib.patches import FancyArrowPatch

def arrow(x1, y1, x2, y2, rad=0.0, lw=1.5, color=ARROW_COLOR, alpha=0.9):
    # Offset endpoints toward each other by half a box so the arrow lives
    # in the gap between boxes, not under them.
    dx, dy = x2 - x1, y2 - y1
    dist = (dx ** 2 + dy ** 2) ** 0.5
    if dist < 1e-6:
        return
    ux, uy = dx / dist, dy / dist
    # Shrink by 0.55 data-units on each side (slightly more than box_w/2 = 0.465)
    SHRINK = 0.56
    sx1, sy1 = x1 + ux * SHRINK, y1 + uy * SHRINK
    sx2, sy2 = x2 - ux * SHRINK, y2 - uy * SHRINK
    ax.add_patch(FancyArrowPatch(
        (sx1, sy1), (sx2, sy2),
        arrowstyle="->", mutation_scale=11,
        linewidth=lw, color=color, alpha=alpha,
        connectionstyle=f"arc3,rad={rad}",
        zorder=2,
    ))

for i in range(len(EXEC_SEQUENCE) - 1):
    x1, lane1 = EXEC_SEQUENCE[i]
    x2, lane2 = EXEC_SEQUENCE[i + 1]
    y1, y2 = LANE_Y[lane1], LANE_Y[lane2]
    same_x = abs(x2 - x1) < 0.05
    # Skip arrows between adjacent same-lane boxes -- left-to-right reading
    # is implicit and the step badges encode the order.  Draw arrows only
    # for: (a) lane changes (visually interesting jumps), and (b) actor->
    # validator pairs at the same x (the actor/reviewer relationship).
    if lane1 == lane2 and not same_x:
        continue
    if same_x and lane1 == lane2:
        continue
    rad = 0.0 if same_x else (-0.22 if y2 < y1 else 0.22)
    arrow(x1, y1, x2, y2, rad=rad, lw=1.5)

# ---------------------------------------------------------------------------
# Helper: draw a phase box, then stamp a step-number badge on the corner
# ---------------------------------------------------------------------------
def draw_box(x, y, label, role, kind, displayed_num=None, step=None):
    if kind == "critic":
        ec, lw, ls, weight = CRITIC_EDGE, 1.8, "-", "bold"
        fc = FILL[role]
        stamp_color = CRITIC_EDGE
    elif kind == "validator":
        ec, lw, ls, weight = EDGE["dataml"], 1.5, "-", "regular"
        fc = VALIDATOR_FILL
        stamp_color = EDGE["dataml"]
    else:
        ec, lw, ls, weight = EDGE[role], 1.5, "-", "regular"
        fc = FILL[role]
        stamp_color = None

    # Background box (filled lane colour)
    ax.add_patch(FancyBboxPatch(
        (x - box_w/2, y - box_h/2), box_w, box_h,
        boxstyle="round,pad=0.03",
        facecolor=fc, edgecolor=ec, linewidth=lw, linestyle=ls,
        zorder=3,
    ))
    # Reviewer "stamp" -- solid colour bar across the top of the box,
    # readable at distance, replacing the diagonal hatch.
    if stamp_color is not None:
        stamp_h = box_h * STAMP_HEIGHT_FRAC
        ax.add_patch(FancyBboxPatch(
            (x - box_w/2 + 0.03, y + box_h/2 - stamp_h - 0.01),
            box_w - 0.06, stamp_h,
            boxstyle="round,pad=0.005",
            facecolor=stamp_color, edgecolor="none",
            zorder=4,
        ))
    # Label text -- slightly below centre so the stamp doesn't overlap
    label_y_offset = -0.04 if stamp_color else 0.0
    ax.text(x, y + label_y_offset, label, ha="center", va="center",
            fontsize=8.0, zorder=5, weight=weight)
    if displayed_num is not None:
        ax.text(x, y + box_h/2 + 0.08, displayed_num,
                ha="center", va="bottom", fontsize=8.5, weight="bold",
                color=EDGE.get("dataml" if kind == "validator" else role, EDGE[role]),
                zorder=4)
    if step is not None:
        # Step badge -- dark filled circle in lower-left corner of the box
        # (lower-left avoids the reviewer top-stamp).
        bx = x - box_w/2 + 0.10
        by = y - box_h/2 + 0.10
        ax.add_patch(Circle((bx, by), 0.15,
                             facecolor=BADGE_FILL, edgecolor="white",
                             linewidth=1.0, zorder=6))
        ax.text(bx, by, str(step), ha="center", va="center",
                fontsize=7.0, color=BADGE_TEXT, weight="bold", zorder=7)

# ---------------------------------------------------------------------------
# Actor / critic boxes
# ---------------------------------------------------------------------------
for x, label, role, kind, num in ACTORS:
    y = LANE_Y["dataml_actor"] if role == "dataml" else LANE_Y["expert"]
    lane_key = "dataml_actor" if role == "dataml" else "expert"
    step = STEP_OF.get((x, lane_key))
    draw_box(x, y, label, role, kind, displayed_num=num, step=step)
    # Coordinator pin (only once per slot to avoid double-draw on Phase 1)
    if not (x == 1 and role == "dataml"):
        ax.plot([x, x], [LANE_Y["coord"] + box_h/2 - 0.5, GATE_Y],
                color="#bbb", linewidth=0.5, linestyle=":", zorder=1)

# ---------------------------------------------------------------------------
# Validator boxes
# ---------------------------------------------------------------------------
for x, vlabel in VALIDATORS:
    step = STEP_OF.get((x, "dataml_val"))
    if x == 22.4:
        draw_box(x, LANE_Y["dataml_val"], vlabel, "dataml", "validator",
                 displayed_num="21", step=step)
        ax.plot([x, x], [LANE_Y["coord"] + box_h/2 - 0.5, GATE_Y],
                color="#bbb", linewidth=0.5, linestyle=":", zorder=1)
    else:
        draw_box(x, LANE_Y["dataml_val"], vlabel, "dataml", "validator",
                 displayed_num=None, step=step)

# ---------------------------------------------------------------------------
# Footer caveat
# ---------------------------------------------------------------------------
ax.text(0.5, -1.20,
        "Numbered badges in each box give execution order across the 34-step "
        "sequence (parallel actors in Phase 1 are listed sequentially).  "
        "Arrows mark lane-changing transitions; same-lane moves follow "
        "left-to-right reading.  Phase 20a (Methods Ledger Refresh) runs "
        "between Phases 19V and 20.  Phase 21 (Deploy Polish) is validator-"
        "only -- its actor and validator are the same frame.",
        ha="left", va="top", fontsize=7.2, color="#555", style="italic")

# ---------------------------------------------------------------------------
# Legend
# ---------------------------------------------------------------------------
# Legend laid out as three columns.  Each column groups a lane's actor with
# its reviewer (or, for the Coordinator column, just the Coordinator entry).
# We build the legend manually with two separate ax.legend calls combined,
# or simpler: use three separate stacked legends positioned side-by-side.
#
# Approach: a single ax.legend with ncol=3 fills row-major
# [0,1,2 / 3,4,5], so to get vertical pairings within each column we order:
#   row 1: LITREVIEW actor    | DATAML actor       | Coordinator
#   row 2: LITREVIEW critic   | DATAML validator   | (blank)
# Custom legend handles that include the top-stamp convention for reviewers,
# matching the actual figure styling.
from matplotlib.legend_handler import HandlerPatch

class StampedPatch(mpatches.Patch):
    """A patch with a colored top stamp -- visual proxy for critic/validator boxes."""
    def __init__(self, fill_color, edge_color, stamp_color, label):
        super().__init__(label=label)
        self._fill = fill_color
        self._edge = edge_color
        self._stamp = stamp_color

class StampedHandler(HandlerPatch):
    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        # Use plain Rectangle for crisp legend swatch rendering
        from matplotlib.patches import Rectangle
        # Base box (lane fill)
        base = Rectangle(
            (-xdescent, -ydescent), width, height,
            facecolor=orig_handle._fill,
            edgecolor=orig_handle._edge,
            linewidth=1.6, transform=trans,
        )
        # Top stamp bar -- top 35% of the swatch, no border, full width
        sh = height * 0.35
        stamp = Rectangle(
            (-xdescent, -ydescent + height - sh), width, sh,
            facecolor=orig_handle._stamp, edgecolor="none",
            transform=trans,
        )
        return [base, stamp]

# matplotlib fills legends column-major by default, so order items
# col-by-col: [col0_top, col0_bot, col1_top, col1_bot, col2_top, col2_bot]
blank = mpatches.Patch(facecolor="none", edgecolor="none", label="")
legend_items = [
    # Column 0: LITREVIEW
    mpatches.Patch(facecolor=FILL["expert"], edgecolor=EDGE["expert"],
                   linewidth=1.4,
                   label="LITREVIEW actor -- scientific phase"),
    StampedPatch(FILL["expert"], CRITIC_EDGE, CRITIC_EDGE,
                 "LITREVIEW critic -- info-barriered review"),
    # Column 1: DATAML
    mpatches.Patch(facecolor=FILL["dataml"], edgecolor=EDGE["dataml"],
                   linewidth=1.4,
                   label="DATAML actor -- mechanical phase"),
    StampedPatch(VALIDATOR_FILL, EDGE["dataml"], EDGE["dataml"],
                 "DATAML validator -- info-barriered review"),
    # Column 2: Coordinator (single entry; blank below for alignment)
    mpatches.Patch(facecolor=FILL["coord"], edgecolor=EDGE["coord"],
                   linewidth=1.4,
                   label="Coordinator -- routing + gates"),
    blank,
]
leg = ax.legend(
    handles=legend_items, labels=[h.get_label() for h in legend_items],
    handler_map={StampedPatch: StampedHandler()},
    loc="lower center", bbox_to_anchor=(0.5, -0.14),
    ncol=3, frameon=False, fontsize=9.0,
    handlelength=2.6, handleheight=1.9,    # taller swatches make stamp readable
    columnspacing=2.5, handletextpad=0.8,
    labelspacing=1.0,
)

ax.set_title(
    "Expert Review Pipeline v29 -- 34-step execution sequence across 21 phases (+ 20a)",
    fontsize=12.5, weight="bold", pad=10,
)

plt.subplots_adjust(left=0.02, right=0.99, top=0.93, bottom=0.13)

out_path = os.path.join(
    os.path.dirname(os.path.abspath("__nb__")), "..", "fig_methods_pipeline.png"
)
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor="white")
print("saved:", out_path)
